In [3]:
import pandas as pd
import re

# ==============================
# 1. LOAD DATA
# ==============================
input_path = "E:/LogAnomalyDetector/data/hdfs_small/HDFS_2k/HDFS_2k.log_structured.csv"
df = pd.read_csv(input_path)

print("Columns:", df.columns)
print("Shape:", df.shape)

# ==============================
# 2. FIND TEXT COLUMN
# ==============================
possible_text_cols = [
    "Content", "message", "log", "text", "line", "event", "description"
]

text_col = None
for col in possible_text_cols:
    if col in df.columns:
        text_col = col
        break

if text_col is None:
    raise ValueError("❌ No suitable text column found")

print(f"Using text column: {text_col}")

df["message"] = df[text_col].astype(str)

# ==============================
# 3. REFINED LABELING FUNCTION
# ==============================
def assign_label(text):
    text = str(text).lower()

    anomaly_keywords = [
        "error", "fail", "exception", "timeout",
        "corrupt", "invalid", "denied",
        "unreachable", "crash", "fatal"
    ]

    anomaly_patterns = [
        r"checksum",
        r"disk",
        r"overflow",
        r"not found",
    ]

    # stricter keyword condition (avoid overfitting)
    keyword_hits = sum(word in text for word in anomaly_keywords)

    if keyword_hits >= 1:
        return 1

    for pattern in anomaly_patterns:
        if re.search(pattern, text):
            return 1

    return 0

df["true_label"] = df["message"].apply(assign_label)

# ==============================
# 4. CREATE TIMESTAMP
# ==============================
df["timestamp"] = range(len(df))

# ==============================
# 5. FINAL DATASET
# ==============================
df_final = df[["timestamp", "message", "true_label"]]

# ==============================
# 6. SAVE
# ==============================
save_path = "E:/LogAnomalyDetector/data/hdfs_processed.csv"
df_final.to_csv(save_path, index=False)

print("\nSaved to:", save_path)
print("Final shape:", df_final.shape)

print("\nLabel Distribution:")
print(df_final["true_label"].value_counts())

print("\nSample:")
print(df_final.head())

Columns: Index(['LineId', 'Date', 'Time', 'Pid', 'Level', 'Component', 'Content',
       'EventId', 'EventTemplate'],
      dtype='object')
Shape: (2000, 9)
Using text column: Content

Saved to: E:/LogAnomalyDetector/data/hdfs_processed.csv
Final shape: (2000, 3)

Label Distribution:
true_label
0    1696
1     304
Name: count, dtype: int64

Sample:
   timestamp                                            message  true_label
0          0  PacketResponder 1 for block blk_38865049064139...           0
1          1  PacketResponder 0 for block blk_-6952295868487...           0
2          2  BLOCK* NameSystem.addStoredBlock: blockMap upd...           0
3          3  PacketResponder 2 for block blk_82291938032499...           0
4          4  PacketResponder 2 for block blk_-6670958622368...           0
